In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA_DIR = "/content/drive/My Drive/Colab Notebooks/Order_Forecasting/Input/shifted_recent"

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), parse_dates=["date"])
stores = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
holidays = pd.read_csv(os.path.join(DATA_DIR, "holidays_events.csv"), parse_dates=["date"])
oil = pd.read_csv(os.path.join(DATA_DIR, "oil.csv"), parse_dates=["date"])

In [ ]:
def add_time_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()
    d["dayofweek"] = d["date"].dt.dayofweek
    d["weekofyear"] = d["date"].dt.isocalendar().week.astype(int)
    d["month"] = d["date"].dt.month
    d["year"] = d["date"].dt.year
    d["day"] = d["date"].dt.day
    d["is_weekend"] = (d["dayofweek"] >= 5).astype(int)
    return d

def add_lag_features(data: pd.DataFrame, group_cols=("store_nbr","family")) -> pd.DataFrame:
    d = data.copy()

    # Ensure proper sorting before lag creation
    d = d.sort_values(list(group_cols) + ["date"])

    # Lags
    for lag in [1, 7, 14, 28]:
        d[f"sales_lag_{lag}"] = (
            d.groupby(list(group_cols))["sales"]
            .transform(lambda x: x.shift(lag))
        )

    # Rolling means (shift first to avoid leakage)
    d["sales_rollmean_7"] = (
        d.groupby(list(group_cols))["sales"]
        .transform(lambda x: x.shift(1).rolling(7).mean())
    )

    d["sales_rollmean_14"] = (
        d.groupby(list(group_cols))["sales"]
        .transform(lambda x: x.shift(1).rolling(14).mean())
    )

    d["sales_rollmean_28"] = (
        d.groupby(list(group_cols))["sales"]
        .transform(lambda x: x.shift(1).rolling(28).mean())
    )

    d["sales_rollstd_28"] = (
        d.groupby(list(group_cols))["sales"]
        .transform(lambda x: x.shift(1).rolling(28).std())
    )

    return d

In [ ]:
df = train.merge(stores, on="store_nbr", how="left")

# holiday flag
holidays["is_holiday"] = 1
hol = holidays[["date", "is_holiday"]].drop_duplicates()
df = df.merge(hol, on="date", how="left")
df["is_holiday"] = df["is_holiday"].fillna(0)

# oil
oil = oil.sort_values("date")
oil["dcoilwtico"] = oil["dcoilwtico"].ffill()
df = df.merge(oil, on="date", how="left")

df_feat = add_time_features(df)
df_feat = add_lag_features(df_feat)

df_feat = df_feat.dropna(subset=["sales_lag_28", "sales_rollmean_28"]).reset_index(drop=True)

In [ ]:
VAL_DAYS = 60

max_date = df_feat["date"].max()
val_start = max_date - pd.Timedelta(days=VAL_DAYS-1)

train_df = df_feat[df_feat["date"] < val_start].copy()
val_df   = df_feat[df_feat["date"] >= val_start].copy()

In [ ]:
cat_cols = ["family", "city", "state", "type", "cluster"]

for c in cat_cols:
    if c in df_feat.columns:
        df_feat[c] = df_feat[c].astype("category")
        categories = df_feat[c].cat.categories
        train_df[c] = pd.Categorical(train_df[c], categories=categories).codes
        val_df[c]   = pd.Categorical(val_df[c], categories=categories).codes

In [ ]:
TARGET = "sales"

drop_cols = ["date", TARGET]

FEATURES = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df[FEATURES]
y_train = np.log1p(train_df[TARGET])

X_val = val_df[FEATURES]
y_val = np.log1p(val_df[TARGET])

In [ ]:
OUTPUT_DIR = "/content/drive/My Drive/Colab Notebooks/Order_Forecasting/Processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train.to_parquet(f"{OUTPUT_DIR}/X_train.parquet")
X_val.to_parquet(f"{OUTPUT_DIR}/X_val.parquet")

pd.Series(y_train).to_frame("target").to_parquet(f"{OUTPUT_DIR}/y_train.parquet")
pd.Series(y_val).to_frame("target").to_parquet(f"{OUTPUT_DIR}/y_val.parquet")

val_df[["date","store_nbr","family","sales"]].to_parquet(f"{OUTPUT_DIR}/val_metadata.parquet")

print("Saved processed datasets successfully.")

Saved processed datasets successfully.
